# AI-CFD MLP Training — Capacity-Matched Colab Version

DGCNN과 모델 용량을 가깝게 맞춘 최종 point-wise MLP baseline 전용 Colab 노트북이다.

최종 MLP 구조:
- `4 → 256 → 256 → 256 → 256 → 2`
- Trainable parameters: `199,170`
- DGCNN trainable parameters: `189,826`

입력:
- `[x, y, z, velocity]`

출력:
- `[HTC, wall_shear]`

Split:
- Train: `face_0001 ~ face_0080`
- Validation: `face_0081 ~ face_0090`
- Test: `face_0091 ~ face_0100`

Google Drive:
- 입력 ZIP: `MyDrive/ai-cfd-flow-prediction/data/05_cfd_csv.zip`
- MLP 결과: `MyDrive/ai-cfd-flow-prediction/mlp/`

이전 25,538-parameter MLP checkpoint가 Drive에 남아 있어도,
Cell 5가 architecture를 검사해서 **구형 checkpoint는 resume하지 않고 새 256×4 학습으로 덮어쓴다.**


## Cell 1 — GPU 확인


In [1]:
!nvidia-smi

import torch

print()
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab 런타임을 T4 GPU로 변경하세요."
    )

print("GPU            :", torch.cuda.get_device_name(0))
print(
    "GPU memory     :",
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "GB",
)


Fri Aug 21 03:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Google Drive 연결


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_MLP = DRIVE_ROOT / "mlp"

DRIVE_ZIP = DRIVE_DATA / "05_cfd_csv.zip"

BEST_MODEL = DRIVE_MLP / "best_model.pt"
SCALER = DRIVE_MLP / "scalers.npz"
LAST_CHECKPOINT = DRIVE_MLP / "last_checkpoint.pt"
TEST_LOG = DRIVE_MLP / "test_evaluation.txt"

DRIVE_MLP.mkdir(
    parents=True,
    exist_ok=True,
)

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(
        f"Google Drive CFD ZIP을 찾을 수 없습니다:\n{DRIVE_ZIP}"
    )

print("=" * 78)
print("GOOGLE DRIVE READY")
print("=" * 78)
print("CSV ZIP         :", DRIVE_ZIP)
print("MLP output      :", DRIVE_MLP)
print("Best model      :", BEST_MODEL)
print("Scaler          :", SCALER)
print("Last checkpoint :", LAST_CHECKPOINT)
print("=" * 78)


Mounted at /content/drive
GOOGLE DRIVE READY
CSV ZIP         : /content/drive/MyDrive/ai-cfd-flow-prediction/data/05_cfd_csv.zip
MLP output      : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp
Best model      : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz
Last checkpoint : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/last_checkpoint.pt


## Cell 3 — 최신 GitHub 코드 + CFD CSV 자동 복구


In [3]:
import shutil
import subprocess
from pathlib import Path

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DATA_ROOT = Path(
    "/content/ai-cfd-data"
)

CSV_DIR = (
    DATA_ROOT
    / "05_cfd_csv"
)

# ================================================================
# 1. GitHub repository
# ================================================================

if REPO.exists():

    print(
        "[Git] Existing repository found -> pull latest main",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "pull",
            "origin",
            "main",
        ],
        check=True,
    )

else:

    print(
        "[Git] Repository not found -> clone",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/hehong01/ai-cfd-flow-prediction.git",
            str(REPO),
        ],
        check=True,
    )

# ================================================================
# 2. CFD CSV restore
# ================================================================

def count_direct_csv():
    if not CSV_DIR.exists():
        return 0
    return len(list(CSV_DIR.glob("*.csv")))

csv_count = count_direct_csv()

if csv_count != 300:

    print(
        f"[Data] Local CSV count = {csv_count} -> restore from Drive",
        flush=True,
    )

    shutil.rmtree(
        DATA_ROOT,
        ignore_errors=True,
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_zip = Path(
        "/content/05_cfd_csv.zip"
    )

    shutil.copy2(
        DRIVE_ZIP,
        local_zip,
    )

    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(local_zip),
            "-d",
            str(DATA_ROOT),
        ],
        check=True,
    )

    csv_count = count_direct_csv()

else:

    print(
        "[Data] Existing 300 local CSV files -> reuse",
        flush=True,
    )

if csv_count != 300:
    raise RuntimeError(
        "CFD CSV 복구 실패:\n"
        f"Expected 300, found {csv_count}\n"
        f"Directory: {CSV_DIR}"
    )

commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "--short",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)

print()
print("=" * 78)
print("COLAB MLP ENVIRONMENT READY")
print("=" * 78)
print("Git commit       :", commit)
print("CSV count        :", csv_count)
print("CSV directory    :", CSV_DIR)
print("Checkpoint exists:", LAST_CHECKPOINT.exists())
print("=" * 78)


[Git] Repository not found -> clone
[Data] Local CSV count = 0 -> restore from Drive

COLAB MLP ENVIRONMENT READY
Git commit       : f369e2e
CSV count        : 300
CSV directory    : /content/ai-cfd-data/05_cfd_csv
Checkpoint exists: False


## Cell 4 — 최종 MLP 구조 self-test

반드시 다음이 보여야 한다.

- `Trainable parameters : 199,170`
- `Parameter count test: PASS`
- `MLP MODEL SELF-TEST PASSED`


In [4]:
%cd /content/ai-cfd-flow-prediction
!python -u 05_model_training/mlp/model.py


/content/ai-cfd-flow-prediction
POINT-WISE MLP SELF-TEST

PointwiseMLP(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=2, bias=True)
  )
)

Trainable parameters : 199,170

[POINT BATCH]
Input shape  : (32, 4)
Output shape : (32, 2)

[GROUPED POINTS]
Input shape  : (2, 7000, 4)
Output shape : (2, 7000, 2)

Parameter count test: PASS
Point input test     : PASS
Grouped input test   : PASS
Finite output test   : PASS

MLP MODEL SELF-TEST PASSED


## Cell 5 — 256×4 MLP 본학습 / 자동 Resume / 실시간 로그

최종 조건:
- Hidden layers: `(256, 256, 256, 256)`
- Parameters: `199,170`
- Batch size: `8192`
- Max epochs: `100`
- Learning rate: `0.001`
- Patience: `15`
- 모든 원본 CFD wall node 사용
- FPS 사용 안 함

구형 25,538-parameter MLP checkpoint가 있으면 architecture mismatch를 감지해
**resume하지 않고 새 학습(`--overwrite`)을 시작한다.**

새 256×4 checkpoint라면 마지막 완료 epoch 다음부터 자동 resume한다.


In [5]:
# ================================================================
# Cell 5 — Capacity-matched MLP full training / resume / live log
# ================================================================

from pathlib import Path
import subprocess
import sys
import torch

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_MLP = (
    DRIVE_ROOT
    / "mlp"
)

DRIVE_MLP.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_SCRIPT = (
    REPO
    / "05_model_training"
    / "mlp"
    / "train.py"
)

BEST_MODEL = (
    DRIVE_MLP
    / "best_model.pt"
)

SCALER = (
    DRIVE_MLP
    / "scalers.npz"
)

LAST_CHECKPOINT = (
    DRIVE_MLP
    / "last_checkpoint.pt"
)

# ================================================================
# Final settings
# ================================================================

TARGET_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 8192
LEARNING_RATE = 1e-3
LOG_EVERY = 50

EXPECTED_HIDDEN_DIMS = (
    256,
    256,
    256,
    256,
)

EXPECTED_PARAMETER_COUNT = 199_170

# ================================================================
# Basic checks
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab 런타임을 T4 GPU로 변경하세요."
    )

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(
        "MLP train.py를 찾을 수 없습니다:\n"
        f"{TRAIN_SCRIPT}"
    )

print("=" * 78, flush=True)
print("CAPACITY-MATCHED MLP FINAL TRAINING LAUNCHER", flush=True)
print("=" * 78, flush=True)
print(
    f"GPU             : {torch.cuda.get_device_name(0)}",
    flush=True,
)
print(
    f"Architecture    : 4 -> 256 -> 256 -> 256 -> 256 -> 2",
    flush=True,
)
print(
    f"Expected params : {EXPECTED_PARAMETER_COUNT:,}",
    flush=True,
)
print(
    f"Batch size      : {BATCH_SIZE}",
    flush=True,
)
print(
    f"Target epochs   : {TARGET_EPOCHS}",
    flush=True,
)
print(
    f"Learning rate   : {LEARNING_RATE}",
    flush=True,
)
print(
    f"Patience        : {PATIENCE}",
    flush=True,
)
print(
    f"Best model      : {BEST_MODEL}",
    flush=True,
)
print(
    f"Scaler          : {SCALER}",
    flush=True,
)
print(
    f"Last checkpoint : {LAST_CHECKPOINT}",
    flush=True,
)
print("=" * 78, flush=True)
print(flush=True)

# ================================================================
# Decide fresh vs resume
# ================================================================

run_training = True
mode = "--overwrite"

if LAST_CHECKPOINT.exists():

    print(
        "Existing MLP checkpoint detected -> architecture check",
        flush=True,
    )

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    checkpoint_hidden_dims = tuple(
        checkpoint.get(
            "hidden_dims",
            (),
        )
    )

    checkpoint_model_name = checkpoint.get(
        "model_name"
    )

    compatible = (
        checkpoint_model_name == "PointwiseMLP"
        and checkpoint_hidden_dims == EXPECTED_HIDDEN_DIMS
    )

    if not compatible:

        print()
        print(
            "[OLD / INCOMPATIBLE CHECKPOINT DETECTED]",
            flush=True,
        )
        print(
            f"Checkpoint model : {checkpoint_model_name}",
            flush=True,
        )
        print(
            f"Checkpoint hidden: {checkpoint_hidden_dims}",
            flush=True,
        )
        print(
            f"Required hidden  : {EXPECTED_HIDDEN_DIMS}",
            flush=True,
        )
        print(
            "구형 MLP 결과는 resume하지 않습니다.",
            flush=True,
        )
        print(
            "새 256×4 MLP를 epoch 1부터 학습하고 "
            "Drive의 MLP 결과를 새 결과로 덮어씁니다.",
            flush=True,
        )

        mode = "--overwrite"

    else:

        if not SCALER.exists():
            raise RuntimeError(
                "호환 checkpoint는 있지만 scalers.npz가 없습니다."
            )

        if not BEST_MODEL.exists():
            raise RuntimeError(
                "호환 checkpoint는 있지만 best_model.pt가 없습니다."
            )

        completed_epoch = int(
            checkpoint["epoch"]
        )

        best_epoch = int(
            checkpoint["best_epoch"]
        )

        best_val_loss = float(
            checkpoint["best_val_loss"]
        )

        no_improve_count = int(
            checkpoint[
                "epochs_without_improvement"
            ]
        )

        checkpoint_batch_size = int(
            checkpoint["batch_size"]
        )

        print()
        print(
            "[COMPATIBLE 256x4 CHECKPOINT]",
            flush=True,
        )
        print(
            f"Completed epoch  : {completed_epoch}",
            flush=True,
        )
        print(
            f"Best epoch       : {best_epoch}",
            flush=True,
        )
        print(
            f"Best val loss    : {best_val_loss:.8f}",
            flush=True,
        )
        print(
            f"No-improve count : {no_improve_count}",
            flush=True,
        )
        print(
            f"Checkpoint batch : {checkpoint_batch_size}",
            flush=True,
        )

        if checkpoint_batch_size != BATCH_SIZE:
            raise RuntimeError(
                "Checkpoint batch size와 현재 설정이 다릅니다.\n"
                f"Checkpoint: {checkpoint_batch_size}\n"
                f"Current   : {BATCH_SIZE}"
            )

        if completed_epoch >= TARGET_EPOCHS:

            print(
                "Training is already complete "
                f"({completed_epoch}/{TARGET_EPOCHS}).",
                flush=True,
            )

            run_training = False

        elif no_improve_count >= PATIENCE:

            print(
                "Training already reached the "
                "early-stopping condition.",
                flush=True,
            )

            run_training = False

        else:

            mode = "--resume"

            print(
                "RESUME TRAINING: "
                f"epoch {completed_epoch + 1}부터 시작합니다.",
                flush=True,
            )

else:

    print(
        "No MLP checkpoint -> 새 256×4 본학습을 시작합니다.",
        flush=True,
    )

print(flush=True)

# ================================================================
# Launch train.py
# ================================================================

if run_training:

    cmd = [
        sys.executable,
        "-u",
        str(TRAIN_SCRIPT),

        "--batch-size",
        str(BATCH_SIZE),

        "--epochs",
        str(TARGET_EPOCHS),

        "--learning-rate",
        str(LEARNING_RATE),

        "--patience",
        str(PATIENCE),

        "--log-every",
        str(LOG_EVERY),

        "--model-path",
        str(BEST_MODEL),

        "--scaler-path",
        str(SCALER),

        "--checkpoint-path",
        str(LAST_CHECKPOINT),

        mode,
    ]

    print("=" * 78, flush=True)
    print("COMMAND", flush=True)
    print("=" * 78, flush=True)
    print(
        " ".join(cmd),
        flush=True,
    )
    print("=" * 78, flush=True)
    print(flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:
        process.kill()
        raise RuntimeError(
            "MLP train.py stdout pipe를 열 수 없습니다."
        )

    try:

        for line in iter(
            process.stdout.readline,
            "",
        ):

            if line == "":
                break

            print(
                line,
                end="",
                flush=True,
            )

    finally:

        process.stdout.close()

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "MLP training failed "
            f"(exit code {return_code})."
        )

    print()
    print("=" * 78)
    print("MLP TRAINING PROCESS FINISHED")
    print("=" * 78)


CAPACITY-MATCHED MLP FINAL TRAINING LAUNCHER
GPU             : Tesla T4
Architecture    : 4 -> 256 -> 256 -> 256 -> 256 -> 2
Expected params : 199,170
Batch size      : 8192
Target epochs   : 100
Learning rate   : 0.001
Patience        : 15
Best model      : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz
Last checkpoint : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/last_checkpoint.pt

No MLP checkpoint -> 새 256×4 본학습을 시작합니다.

COMMAND
/usr/bin/python3 -u /content/ai-cfd-flow-prediction/05_model_training/mlp/train.py --batch-size 8192 --epochs 100 --learning-rate 0.001 --patience 15 --log-every 50 --model-path /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt --scaler-path /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz --checkpoint-path /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/last_checkpoint.pt --overwrite

POINT-WISE MLP TRAINING
Data root        

## Cell 6 — 최종 held-out TEST 평가

학습 종료 후 실행한다.

- Test faces: `face_0091 ~ face_0100`
- 30 CSV
- 모든 원본 test wall node 사용
- 결과는 `mlp/test_evaluation.txt`에 저장


In [6]:
from pathlib import Path
import subprocess
import sys

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DRIVE_MLP = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/mlp"
)

BEST_MODEL = DRIVE_MLP / "best_model.pt"
SCALER = DRIVE_MLP / "scalers.npz"
TEST_LOG = DRIVE_MLP / "test_evaluation.txt"

EVAL_SCRIPT = (
    REPO
    / "05_model_training"
    / "mlp"
    / "evaluate.py"
)

if not BEST_MODEL.exists():
    raise FileNotFoundError(BEST_MODEL)

if not SCALER.exists():
    raise FileNotFoundError(SCALER)

cmd = [
    sys.executable,
    "-u",
    str(EVAL_SCRIPT),
    "--batch-size",
    "8192",
    "--model-path",
    str(BEST_MODEL),
    "--scaler-path",
    str(SCALER),
]

print("=" * 78)
print("CAPACITY-MATCHED MLP HELD-OUT TEST")
print("=" * 78)
print(" ".join(cmd))
print("=" * 78)
print()

with TEST_LOG.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:
        process.kill()
        raise RuntimeError(
            "MLP evaluate.py stdout pipe를 열 수 없습니다."
        )

    try:

        for line in iter(
            process.stdout.readline,
            "",
        ):

            if line == "":
                break

            print(
                line,
                end="",
                flush=True,
            )

            log_file.write(
                line
            )

            log_file.flush()

    finally:

        process.stdout.close()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"MLP evaluation failed (exit code {return_code})."
    )

print()
print("Evaluation log saved:", TEST_LOG)


CAPACITY-MATCHED MLP HELD-OUT TEST
/usr/bin/python3 -u /content/ai-cfd-flow-prediction/05_model_training/mlp/evaluate.py --batch-size 8192 --model-path /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt --scaler-path /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz

POINT-WISE MLP TEST EVALUATION
Data root      : /content/ai-cfd-data
CSV directory  : /content/ai-cfd-data/05_cfd_csv
Model          : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt
Scaler         : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz
Device         : cuda
GPU            : Tesla T4
PyTorch        : 2.11.0+cu128
Batch size     : 8192

[LOAD SCALER]
CFD SCALER STATISTICS

[INPUT]
x            mean = -0.0009643902173   std =  0.03485832641
y            mean =  0.003779579762   std =  0.05201426341
z            mean = -0.02745688454   std =  0.04530750024
velocity     mean =  7.666666667   std =  2.054804668

[TARGET]
HTC          mean =  54.05260697   std =  

## Cell 7 — 최종 결과 요약


In [7]:
from pathlib import Path
import torch

DRIVE_MLP = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/mlp"
)

BEST_MODEL = DRIVE_MLP / "best_model.pt"
LAST_CHECKPOINT = DRIVE_MLP / "last_checkpoint.pt"
SCALER = DRIVE_MLP / "scalers.npz"
TEST_LOG = DRIVE_MLP / "test_evaluation.txt"

EXPECTED_HIDDEN_DIMS = (
    256,
    256,
    256,
    256,
)

print("=" * 78)
print("FINAL CAPACITY-MATCHED MLP RESULT SUMMARY")
print("=" * 78)

if not BEST_MODEL.exists():
    raise FileNotFoundError(BEST_MODEL)

best = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

best_hidden_dims = tuple(
    best.get(
        "hidden_dims",
        (),
    )
)

if best_hidden_dims != EXPECTED_HIDDEN_DIMS:
    raise RuntimeError(
        "최종 best_model.pt가 256×4 MLP가 아닙니다.\n"
        f"Found: {best_hidden_dims}"
    )

print()
print("[BEST MODEL]")
print("Best epoch       :", best.get("epoch"))
print("Best val loss    :", best.get("val_loss"))
print("Hidden dims      :", best_hidden_dims)
print("Parameter target :", "199,170")

if LAST_CHECKPOINT.exists():

    last = torch.load(
        LAST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    print()
    print("[LAST TRAINING STATE]")
    print("Completed epoch  :", last.get("epoch"))
    print("Best epoch       :", last.get("best_epoch"))
    print("Best val loss    :", last.get("best_val_loss"))
    print(
        "No-improve count :",
        last.get("epochs_without_improvement"),
    )

print()
print("[FILES]")

for path in (
    BEST_MODEL,
    LAST_CHECKPOINT,
    SCALER,
    TEST_LOG,
):

    if path.exists():

        print(
            f"{path.name:24s} "
            f"{path.stat().st_size:,} bytes"
        )

print()
print("=" * 78)
print("HELD-OUT TEST RESULT")
print("=" * 78)

if TEST_LOG.exists():

    print(
        TEST_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

else:

    print(
        "test_evaluation.txt가 아직 없습니다. "
        "Cell 6을 먼저 실행하세요."
    )

print("=" * 78)


FINAL CAPACITY-MATCHED MLP RESULT SUMMARY

[BEST MODEL]
Best epoch       : 6
Best val loss    : 0.17470112833350446
Hidden dims      : (256, 256, 256, 256)
Parameter target : 199,170

[LAST TRAINING STATE]
Completed epoch  : 21
Best epoch       : 6
Best val loss    : 0.17470112833350446
No-improve count : 15

[FILES]
best_model.pt            801,317 bytes
last_checkpoint.pt       2,403,685 bytes
scalers.npz              1,126 bytes
test_evaluation.txt      3,633 bytes

HELD-OUT TEST RESULT
POINT-WISE MLP TEST EVALUATION
Data root      : /content/ai-cfd-data
CSV directory  : /content/ai-cfd-data/05_cfd_csv
Model          : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/best_model.pt
Scaler         : /content/drive/MyDrive/ai-cfd-flow-prediction/mlp/scalers.npz
Device         : cuda
GPU            : Tesla T4
PyTorch        : 2.11.0+cu128
Batch size     : 8192

[LOAD SCALER]
CFD SCALER STATISTICS

[INPUT]
x            mean = -0.0009643902173   std =  0.03485832641
y            mean =  